# 레이어별 민감도 분석 + 최적 보호 레이어 탐색

## 목적
30개 레이어 중 양자화에 가장 민감한 레이어를 데이터 기반으로 찾아 최적 FP16 보호 조합 도출.

## 방법
1. **Phase 1**: 각 레이어 i를 개별적으로 양자화하여 perplexity 변화 측정 (30회)
2. **Phase 2**: 민감도 순위 기반 Top-k 조합 테스트
3. **Phase 3**: 최적 조합으로 최종 모델 생성

## 핵심 인사이트
- V13은 L0+L29를 보호했지만, 이것이 최적인지 검증 필요
- 보호 레이어가 적을수록 SpeedNorm 유리 (Marlin 커널이 INT4에서 가장 빠름)
- PerfNorm 향상이 SpeedNorm 손실을 상회하는 최소 보호 세트 찾기

---

# 1. Import

In [ ]:
import os
import gc
import json
import torch
import shutil
import numpy as np
from pathlib import Path
from copy import deepcopy

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("\n\u2705 Import 완료!")

# 2. 설정

In [ ]:
# ============================================================================
# 공통 설정
# ============================================================================
MODEL_ID = "./open/base_model"
DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"
NUM_LAYERS = 30

# V13 최적 설정 유지
SCHEME = "W4A16"
TARGETS = ["Linear"]
BLOCK_SIZE = 128
DAMPENING_FRAC = 0.001
ACTORDER = "weight"

# 민감도 분석용 빠른 설정 (Perplexity 측정용)
SENSITIVITY_SAMPLES = 64      # 분석용: 적은 샘플로 빠르게
SENSITIVITY_SEQ_LEN = 512
PPL_EVAL_SAMPLES = 32         # perplexity 측정용 샘플

# 최종 양자화용 설정 (V13과 동일)
FINAL_SAMPLES = 256
FINAL_SEQ_LEN = 512

ORIGINAL_MODEL_SIZE_GB = 2.56

print("=" * 60)
print("레이어 민감도 분석 설정")
print("=" * 60)
print(f"총 레이어 수: {NUM_LAYERS}")
print(f"민감도 분석: {SENSITIVITY_SAMPLES} samples, {SENSITIVITY_SEQ_LEN} seq_len")
print(f"PPL 평가: {PPL_EVAL_SAMPLES} samples")
print(f"최종 양자화: {FINAL_SAMPLES} samples, {FINAL_SEQ_LEN} seq_len")
print("=" * 60)

# 3. 데이터 로드

In [ ]:
print("[INFO] 토크나이저 로드...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

print("[INFO] 캘리브레이션 데이터 로드...")
ds_full = load_dataset(
    DATASET_ID,
    split=f"{DATASET_SPLIT}[:500]",  # 넉넉히 로드
)

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False
        )
    }

ds_full = ds_full.map(preprocess)

# 민감도 분석용 / PPL 평가용 / 최종 양자화용 분리
ds_sensitivity = ds_full.select(range(SENSITIVITY_SAMPLES))
ds_ppl_eval = ds_full.select(range(SENSITIVITY_SAMPLES, SENSITIVITY_SAMPLES + PPL_EVAL_SAMPLES))
ds_final = ds_full.select(range(FINAL_SAMPLES))

print(f"[INFO] 민감도 분석 데이터: {len(ds_sensitivity)} samples")
print(f"[INFO] PPL 평가 데이터: {len(ds_ppl_eval)} samples")
print(f"[INFO] 최종 양자화 데이터: {len(ds_final)} samples")

# 4. Phase 1: Perplexity 기반 민감도 측정

각 레이어를 개별적으로 W4A16 양자화하고, 나머지는 FP16 유지.
양자화 후 perplexity 증가가 큰 레이어 = 민감한 레이어.

In [ ]:
def compute_perplexity(model, tokenizer, dataset, max_length=512):
    """간단한 perplexity 계산."""
    model.eval()
    total_loss = 0
    total_tokens = 0
    
    device = next(model.parameters()).device
    
    with torch.no_grad():
        for example in dataset:
            inputs = tokenizer(
                example["text"],
                return_tensors="pt",
                max_length=max_length,
                truncation=True,
            ).to(device)
            
            outputs = model(**inputs, labels=inputs["input_ids"])
            total_loss += outputs.loss.item() * inputs["input_ids"].shape[1]
            total_tokens += inputs["input_ids"].shape[1]
    
    avg_loss = total_loss / total_tokens
    perplexity = torch.exp(torch.tensor(avg_loss)).item()
    return perplexity

print("\u2705 Perplexity 함수 정의 완료")

In [ ]:
# 베이스라인 perplexity (원본 FP16 모델)
print("[INFO] 원본 모델 로드 중...")

if torch.cuda.is_available():
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, dtype=torch.bfloat16, trust_remote_code=True
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, dtype=torch.float32, trust_remote_code=True, device_map="cpu"
    )

print("[INFO] 베이스라인 perplexity 측정 중...")
baseline_ppl = compute_perplexity(base_model, tokenizer, ds_ppl_eval)
print(f"[INFO] 베이스라인 PPL: {baseline_ppl:.4f}")

# 메모리 해제
del base_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# 레이어별 민감도 측정
# 방법: 모든 레이어를 양자화하되, 레이어 i만 FP16 유지
# → PPL이 낮을수록 = 레이어 i가 민감 (보호 시 PPL 감소가 큼)

results = {}
results_file = "./sensitivity_results.json"

# 이전 결과 로드 (중단 후 재개 가능)
if os.path.exists(results_file):
    with open(results_file) as f:
        results = json.load(f)
    print(f"[INFO] 이전 결과 로드: {len(results)}개 레이어 완료")

print("\n" + "=" * 60)
print("Phase 1: 레이어별 민감도 측정")
print("=" * 60)

for layer_idx in range(NUM_LAYERS):
    key = str(layer_idx)
    if key in results:
        print(f"  Layer {layer_idx:2d}: PPL={results[key]['ppl']:.4f} (캐시)")
        continue
    
    print(f"\n[{layer_idx+1}/{NUM_LAYERS}] Layer {layer_idx} 분석 중...")
    
    # 모델 다시 로드 (각 실험마다 fresh)
    if torch.cuda.is_available():
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, dtype=torch.bfloat16, trust_remote_code=True
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, dtype=torch.float32, trust_remote_code=True, device_map="cpu"
        )
    
    # 레이어 i만 FP16 유지 (나머지 전부 양자화)
    ignore_list = [
        "embed_tokens", "lm_head",
        f"re:model\\.layers\\.{layer_idx}\\..*",
    ]
    
    recipe = [
        GPTQModifier(
            scheme=SCHEME,
            targets=TARGETS,
            ignore=ignore_list,
            block_size=BLOCK_SIZE,
            dampening_frac=DAMPENING_FRAC,
            actorder=ACTORDER,
        )
    ]
    
    oneshot(
        model=model,
        dataset=ds_sensitivity,
        recipe=recipe,
        max_seq_length=SENSITIVITY_SEQ_LEN,
        num_calibration_samples=SENSITIVITY_SAMPLES,
    )
    
    ppl = compute_perplexity(model, tokenizer, ds_ppl_eval)
    results[key] = {"ppl": ppl, "layer": layer_idx}
    print(f"  Layer {layer_idx:2d}: PPL={ppl:.4f} (delta={ppl - baseline_ppl:+.4f})")
    
    # 중간 저장
    with open(results_file, "w") as f:
        json.dump(results, f, indent=2)
    
    # 메모리 해제
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n\u2705 Phase 1 완료!")

# 5. 민감도 순위 분석

In [ ]:
# 결과 로드 및 분석
with open(results_file) as f:
    results = json.load(f)

print(f"베이스라인 PPL: {baseline_ppl:.4f}")
print("\n" + "=" * 60)
print("레이어별 민감도 (PPL 낮은 순 = 해당 레이어 보호 시 가장 효과적)")
print("=" * 60)

# PPL이 낮을수록 해당 레이어를 보호하는 것이 효과적
sorted_layers = sorted(results.items(), key=lambda x: x[1]["ppl"])

print(f"\n{'순위':>4} | {'레이어':>6} | {'PPL':>10} | {'Delta':>10} | 비고")
print("-" * 55)

for rank, (key, data) in enumerate(sorted_layers, 1):
    layer = data["layer"]
    ppl = data["ppl"]
    delta = ppl - baseline_ppl
    note = ""
    if layer == 0:
        note = "← V13 보호"
    elif layer == 29:
        note = "← V13 보호"
    print(f"{rank:4d} | L{layer:>4d} | {ppl:>10.4f} | {delta:>+10.4f} | {note}")

# Top-5 민감 레이어
top5 = [int(k) for k, _ in sorted_layers[:5]]
print(f"\n\u2b50 Top-5 민감 레이어 (보호 추천): {top5}")
print(f"\u2b50 V13 보호 레이어: [0, 29]")

# 6. Phase 2: 최적 조합 테스트

민감도 순위를 바탕으로 다양한 보호 조합을 테스트.

In [ ]:
# 테스트할 보호 조합 정의
# sorted_layers 기반으로 자동 생성 + 수동 추가
top1 = int(sorted_layers[0][0])
top2 = [int(sorted_layers[0][0]), int(sorted_layers[1][0])]
top3 = [int(sorted_layers[0][0]), int(sorted_layers[1][0]), int(sorted_layers[2][0])]

COMBINATIONS = {
    "none": [],                   # V6 (보호 없음)
    "L29_only": [29],             # L29만 보호
    "L0_only": [0],               # L0만 보호
    "V13": [0, 29],               # V13 (L0+L29)
    "top1": [top1],               # 민감도 1위만
    "top2": sorted(top2),         # 민감도 1-2위
    "top3": sorted(top3),         # 민감도 1-3위
}

print("테스트할 조합:")
for name, layers in COMBINATIONS.items():
    print(f"  {name}: {layers}")

In [ ]:
# 각 조합에 대해 양자화 + PPL 측정
combo_results = {}
combo_file = "./combo_results.json"

if os.path.exists(combo_file):
    with open(combo_file) as f:
        combo_results = json.load(f)
    print(f"[INFO] 이전 결과 로드: {len(combo_results)}개 조합")

for name, protect_layers in COMBINATIONS.items():
    if name in combo_results:
        print(f"  {name}: PPL={combo_results[name]['ppl']:.4f} (캐시)")
        continue
    
    print(f"\n[{name}] 보호 레이어: {protect_layers}")
    
    # 모델 로드
    if torch.cuda.is_available():
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, dtype=torch.bfloat16, trust_remote_code=True
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, dtype=torch.float32, trust_remote_code=True, device_map="cpu"
        )
    
    ignore_list = ["embed_tokens", "lm_head"]
    for li in protect_layers:
        ignore_list.append(f"re:model\\.layers\\.{li}\\..*")
    
    recipe = [
        GPTQModifier(
            scheme=SCHEME,
            targets=TARGETS,
            ignore=ignore_list,
            block_size=BLOCK_SIZE,
            dampening_frac=DAMPENING_FRAC,
            actorder=ACTORDER,
        )
    ]
    
    oneshot(
        model=model,
        dataset=ds_sensitivity,
        recipe=recipe,
        max_seq_length=SENSITIVITY_SEQ_LEN,
        num_calibration_samples=SENSITIVITY_SAMPLES,
    )
    
    ppl = compute_perplexity(model, tokenizer, ds_ppl_eval)
    
    # 모델 크기 추정 (보호 레이어 수에 비례)
    # 각 FP16 레이어 ≈ 71MB, W4A16 레이어 ≈ 20MB, delta ≈ 51MB
    base_size_mb = 1420  # V6 기준 (0 protected)
    estimated_size_mb = base_size_mb + len(protect_layers) * 51
    
    combo_results[name] = {
        "ppl": ppl,
        "protect_layers": protect_layers,
        "n_protected": len(protect_layers),
        "estimated_size_mb": estimated_size_mb,
    }
    
    print(f"  PPL={ppl:.4f}, 추정 크기={estimated_size_mb}MB")
    
    with open(combo_file, "w") as f:
        json.dump(combo_results, f, indent=2)
    
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n\u2705 Phase 2 완료!")

In [ ]:
# 결과 비교
print("=" * 70)
print("조합별 결과 비교")
print("=" * 70)
print(f"{'조합':>10} | {'보호 레이어':>15} | {'PPL':>10} | {'크기(MB)':>10} | {'PPL/크기 효율':>12}")
print("-" * 70)

for name, data in sorted(combo_results.items(), key=lambda x: x[1]["ppl"]):
    layers_str = str(data["protect_layers"])
    ppl = data["ppl"]
    size = data["estimated_size_mb"]
    # PPL 감소 대비 크기 증가 효율
    none_ppl = combo_results.get("none", {}).get("ppl", ppl)
    if size > 1420:
        efficiency = (none_ppl - ppl) / (size - 1420) * 1000 if size > 1420 else 0
    else:
        efficiency = 0
    print(f"{name:>10} | {layers_str:>15} | {ppl:>10.4f} | {size:>10} | {efficiency:>12.4f}")

# 최적 조합 추천
best = min(combo_results.items(), key=lambda x: x[1]["ppl"])
print(f"\n\u2b50 최저 PPL 조합: {best[0]} (layers={best[1]['protect_layers']}, PPL={best[1]['ppl']:.4f})")

# 1개 레이어만 보호하는 것 중 최적
single_layer = {k: v for k, v in combo_results.items() if v["n_protected"] == 1}
if single_layer:
    best_single = min(single_layer.items(), key=lambda x: x[1]["ppl"])
    print(f"\u2b50 1개 레이어 최적: {best_single[0]} (PPL={best_single[1]['ppl']:.4f})")

# 7. Phase 3: 최적 조합으로 최종 모델 생성

위 분석 결과를 보고 최적 조합을 선택한 후 아래에서 최종 양자화를 실행.

In [ ]:
# ============================================================================
# ⭐ 최적 조합 선택 (Phase 2 결과를 보고 수정)
# ============================================================================
BEST_COMBO = "V13"  # <<< Phase 2 결과를 보고 최적 조합명 입력
PROTECT_LAYERS = combo_results[BEST_COMBO]["protect_layers"]

OUT_DIR = "./model_optimal"

print(f"최적 조합: {BEST_COMBO}")
print(f"보호 레이어: {PROTECT_LAYERS}")
print(f"출력 디렉토리: {OUT_DIR}")

In [ ]:
# 최종 양자화 (전체 캘리브레이션 데이터 사용)
print("[INFO] 최종 모델 양자화 시작")

if torch.cuda.is_available():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, dtype=torch.bfloat16, trust_remote_code=True
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, dtype=torch.float32, trust_remote_code=True, device_map="cpu"
    )

ignore_list = ["embed_tokens", "lm_head"]
for li in PROTECT_LAYERS:
    ignore_list.append(f"re:model\\.layers\\.{li}\\..*")

recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=ignore_list,
        block_size=BLOCK_SIZE,
        dampening_frac=DAMPENING_FRAC,
        actorder=ACTORDER,
    )
]

oneshot(
    model=model,
    dataset=ds_final,
    recipe=recipe,
    max_seq_length=FINAL_SEQ_LEN,
    num_calibration_samples=FINAL_SAMPLES,
)

print("\n[INFO] 최종 양자화 완료!")

In [ ]:
# 저장
print(f"[INFO] 모델 저장: {OUT_DIR}")

if os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

total_size = 0
for f in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(os.path.join(OUT_DIR, f))
    total_size += size
    print(f"  {f}: {size/1e6:.1f} MB")

quantized_size_gb = total_size / 1e9
print(f"\n모델 크기: {quantized_size_gb:.2f} GB (원본: {ORIGINAL_MODEL_SIZE_GB} GB)")

In [ ]:
# 제출 파일 생성
submit_dir = "./submit"
os.makedirs(submit_dir, exist_ok=True)

zip_name = "submit_optimal"
zip_path = os.path.join(submit_dir, zip_name)

if os.path.exists(f"{zip_path}.zip"):
    os.remove(f"{zip_path}.zip")

shutil.make_archive(
    base_name=zip_path,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,
)

zip_size = os.path.getsize(f"{zip_path}.zip") / 1e9
print(f"\u2705 {zip_path}.zip ({zip_size:.2f} GB)")